## Homework 3: Visualizations of Keplerian orbits

<strong>Wherever possible, use <tt>scipy.constants</tt> instead of programming in constants by hand.</strong> You can find the masses and radii of solar system planets and the sun at https://sites.astro.caltech.edu/~george/constants.html.

Hint: There are multiple ways to draw ellipses in python. One way is to import an ellipse-drawing function from matplotlib.patches.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy import constants

plt.rcParams.update({'font.size':16, 'axes.labelsize':16})

#### Satellite orbit

An Earth satellite is observed at perigee (closest point to Earth) 250 km above Earth's surface. At apogee, it is 3000 km from Earth's surface.

<ol>
    <li>Calculate the eccentricity of the orbit. Remember that Earth is not a point mass.</li>
    <li>Plot the figure of the orbit (i.e. shape of the ellipse) to scale. Mark both foci.</li>
</ol>

#### Planet hunting

One way to discover extrasolar planets is to monitor the line-of-sight velocity (called radial velocity or RV) of a nearby star. A periodic signal in the RV could indicate that the star has a planet. In order to model the planet's orbit, the astronomers have to be able to solve Kepler's equation: 

#### $M = E - e \sin E$ 

where $M$ is the mean anomaly, $E$ is the eccentric anomaly, and $e$ is the eccentricity. Since Kepler's equation is transcendental, it has no analytical solution and must be solved numerically.

<ol start=3>
    <li>The true anomaly $f(t)$ (where $t$ is time) is the angle between ellipse's major axis and the vector connecting the star to the planet. In a markdown cell, describe how $E(t)$ and $M(t)$ relate to $f(t)$.</li>
    <li>The following cell contains code for solving Kepler's equation and computing the radial velocity as a function of time:</li>
</ol>

In [ ]:
# To solve Kepler's equation, make a good starting guess for eccentric anomaly
def KeplerStart3(e, M):
    # Inputs: eccentricity, mean anomaly in RADIANS
    t34 = e**2
    t35 = e*t34
    t33 = np.cos(M)
    return M + (-0.5*t35 + e + (t34 + 1.5*t33*t35)*t33) * np.sin(M)

# 3rd-order method for improving the guess for eccentric anomaly
def eps3(e, M, x):
    # Inputs: eccentricity, mean anomaly in RADIANS, estimate for eccentric anomaly in RADIANS
    t1 = np.cos(x)
    t2 = -1 + e*t1
    t3 = np.sin(x)
    t4 = e*t3
    t5 = -x + t4 + M
    t6 = t5/(0.5*t5*t4/t2 + t2)
    return t5 / ((0.5*t3 - (1./6.)*t1*t6)*e*t6 + t2);

# Solve Kepler's equation (suggested tolerance is 1e-14; Ford (2009) suggests 1e-6 is 
# good enough)
def KeplerSolve(e, M, tol):
    # Inputs: eccentricity, mean anomaly in RADIANS, tolerance
    Mnorm = M % (2*np.pi)
    E0 = KeplerStart3(e, Mnorm)
    dE = tol + 1
    count = 0
    while (dE > tol):
        EE = E0 - eps3(e, Mnorm, E0)
        dE = np.abs(EE - E0)
        E0 = EE
        count += 1
        if (count == 100):
            print("Astounding! KeplerSolve failed to converge!")
            break
    return EE

# Compute RV
def rv(inpars, e, omega, M0, Msini):
    # Inputs: inpars is a DICTIONARY of {"Tobs": observation times in SECONDS 
    # shifted so tobs[0] = 0, "Period": period in SECONDS, "Mstar": star mass in KG}. 
    # (The dictionary structure is because we don't want the Keplergram algorithm 
    # to optimize period or Mstar.) 
    # Other inputs are eccentricity, argument of periastron in RADIANS, 
    # mean anomaly at t=0 in RADIANS, planet M sin i in KG, RV offset in M/S
    # tol = 1.0e-10 # tolerance for KeplerSolve
    tobs = inpars["Tobs"]    
    period = inpars["Period"]
    mstar = inpars["Mstar"]
    am = np.cbrt(constants.G * mstar * period**2. / (4.*np.pi**2.)) # semimajor axis in meters
    # Pre-compute factor needed to convert eccentric anomaly to mean anomaly
    efactor = np.sqrt((1.+e)/(1.-e))
    # Turn observation times into mean anomalies
    Mvec = 2.*np.pi*(tobs/period) + M0
    # Find the eccentric anomaly using KeplerSolve, then the true anomaly analytically
    ftrue = np.zeros_like(Mvec)
    for i in range(len(tobs)):
        EE = KeplerSolve(e, Mvec[i], 1.0e-10)
        ftrue[i] = 2. * np.arctan(efactor*np.tan(0.5*EE))
    # Finally, return the radial velocity
    return np.sqrt(constants.G / ((mstar+Msini)*am*(1.-e**2.))) * Msini * \
        (np.cos(omega + ftrue) + e*np.cos(omega))

Choose any planet planet from the <a href="https://exoplanet.eu/catalog/">Exoplanet Encyclopaedia</a> that has estimates for the mass, period, and eccentricity ($e$). You do not need an estimate of the planet radius. Calculate and plot the star's radial velocity as a function of time for at least three planet orbits. Put the name of your planet somewhere on your plot. Put a link to the planet's Exoplanet Encyclopaedia page in a markdown cell below.

Hints:
<ul>
<li>Unless you want to hunt down the actual observation times from the published literature on your star, you will need to make a vector of synthetic "observation times" to put into the dictionary that is the first argument of <tt>rv()</tt>. You can start at $t = 0$ and use <tt>np.linspace()</tt> or <tt>np.arange()</tt> to get a time vector that covers three or more orbital periods.</li> 
<li><tt>rv()</tt> requires a value for $\omega$, the argument of periastron of the orbit. $\omega$ is not in the Exoplanet Encyclopaedia table, but you can find it if you click on the name of the planet in the leftmost column. Make sure to convert from degrees to radians.</li>
<li>The first argument to <tt>rv()</tt> is a dictionary that packages the observation times, orbital period, and star mass. Star masses are also not in the planet table, but you will find them in the detailed info that you get after clicking on the planet name.</li>
<li><tt>Msini</tt> is the planet mass multiplied with the sine of the orbit inclination. If your chosen planet has no entry under $i$ (inclination) in the Exoplanet Encyclopaedia table, the number in the Mass column is actually $M \sin i$, which is a minimum mass estimate. The true mass could be higher. Often $i$ is unknown, but sometimes it can be measured (i.e. from transits).</li>
<li>Be sure to convert all numbers that you get from the Exoplanet Encyclopaedia into mks units. $M_{\rm Jup}$ is the mass of Jupiter.</li>
</ul>